# Synthetic Labeled Data Generation for RAG Evaluation

This notebook generates ~1000 labeled samples using LLM-as-judge to evaluate RAG outputs.

**Output Format**:
- `question`: The question
- `answer`: Generated answer
- `context`: Retrieved contexts
- `label`: 0 (incorrect) or 1 (correct)
- `confidence`: Judge confidence (0.0-1.0)
- `reasoning`: Judge explanation

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import os
import pandas as pd
import re
import asyncio
from asyncio import Semaphore
from pathlib import Path
from tqdm.auto import tqdm
import dotenv
import google.generativeai as genai

# Load environment variables
dotenv.load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment")

genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

print("✅ Setup complete")

## 2. Load RAG Predictions

In [ ]:
# Load predictions from Normal RAG baseline
predictions_path = "../results/asqa_normal_rag_predictions.csv"

if not Path(predictions_path).exists():
    raise FileNotFoundError(
        f"Predictions file not found: {predictions_path}\n"
        "Please run rag-asqa-baseline.ipynb first to generate predictions."
    )

df_predictions = pd.read_csv(predictions_path)
print(f"Loaded {len(df_predictions)} predictions")
print(f"\nColumns: {df_predictions.columns.tolist()}")
df_predictions.head()

## 3. Sample Selection Strategy

To get ~1000 high-quality labeled samples:
1. Sample ~1200 examples from training set (to account for filtering)
2. Generate diverse examples with different retrieval settings
3. Filter low-confidence labels

In [ ]:
# Sample 1200 examples
import random
random.seed(42)

sample_size = min(1200, len(df_predictions))
df_sample = df_predictions.sample(n=sample_size, random_state=42)

print(f"Sampled {len(df_sample)} examples for labeling")

## 4. LLM Judge Prompt and Labeling

In [ ]:
JUDGE_PROMPT_TEMPLATE = """You are an expert evaluator for question-answering systems.

Question: {question}
Generated Answer: {answer}
Retrieved Context: {contexts}

Evaluate if the answer is CORRECT based on:
1. Factual accuracy (grounded in context)
2. Completeness (addresses all aspects of question)
3. Relevance (directly answers the question)
4. No hallucinations (no information outside context)

Provide your evaluation in this exact format:
Label: [0 or 1]
Confidence: [0.0-1.0]
Reasoning: [brief explanation in 1-2 sentences]

Where:
- Label 1 = CORRECT answer (factually accurate, complete, relevant, no hallucinations)
- Label 0 = INCORRECT answer (factual errors, incomplete, irrelevant, or hallucinations)
- Confidence = how confident you are in this judgment (0.0 = not confident, 1.0 = very confident)
"""

def parse_judge_response(response_text: str) -> tuple:
    """
    Parse LLM judge response to extract label, confidence, and reasoning.
    
    Returns:
        Tuple of (label, confidence, reasoning)
    """
    label = 0
    confidence = 0.0
    reasoning = ""
    
    # Extract label
    label_match = re.search(r'Label:\s*(\d)', response_text, re.IGNORECASE)
    if label_match:
        label = int(label_match.group(1))
    
    # Extract confidence
    conf_match = re.search(r'Confidence:\s*(\d+\.?\d*)', response_text, re.IGNORECASE)
    if conf_match:
        confidence = float(conf_match.group(1))
    
    # Extract reasoning
    reasoning_match = re.search(r'Reasoning:\s*(.+?)(?:\n|$)', response_text, re.IGNORECASE | re.DOTALL)
    if reasoning_match:
        reasoning = reasoning_match.group(1).strip()
    
    return label, confidence, reasoning

# Test parsing
test_response = """Label: 1
Confidence: 0.85
Reasoning: The answer is factually accurate and addresses all aspects of the question."""

label, conf, reason = parse_judge_response(test_response)
print(f"Test parsing: label={label}, confidence={conf}, reasoning={reason}")

## 5. Async Batch Labeling

In [ ]:
async def label_example_async(
    question: str,
    answer: str,
    contexts: list,
    semaphore: Semaphore
) -> tuple:
    """
    Asynchronously label a single example using LLM judge.
    
    Returns:
        Tuple of (label, confidence, reasoning)
    """
    async with semaphore:
        try:
            # Format contexts
            if isinstance(contexts, str):
                contexts_str = contexts
            elif isinstance(contexts, list):
                contexts_str = "\n\n".join([f"[{i+1}] {ctx[:300]}..." for i, ctx in enumerate(contexts[:5])])
            else:
                contexts_str = str(contexts)
            
            prompt = JUDGE_PROMPT_TEMPLATE.format(
                question=question[:500],
                answer=answer[:1000],
                contexts=contexts_str[:1500]
            )
            
            loop = asyncio.get_event_loop()
            response = await loop.run_in_executor(
                None,
                lambda: model.generate_content(prompt)
            )
            
            label, confidence, reasoning = parse_judge_response(response.text)
            return label, confidence, reasoning
        
        except Exception as e:
            print(f"Error labeling example: {e}")
            return 0, 0.0, f"Error: {str(e)}"


async def label_batch_async(
    df: pd.DataFrame,
    max_concurrent: int = 10
) -> list:
    """
    Label a batch of examples asynchronously.
    
    Returns:
        List of dictionaries with labeled data
    """
    semaphore = Semaphore(max_concurrent)
    labeled_data = []
    
    tasks = []
    for idx, row in df.iterrows():
        task = label_example_async(
            question=row['question'],
            answer=row['predicted_answer'],
            contexts=row.get('contexts', []),
            semaphore=semaphore
        )
        tasks.append((idx, row, task))
    
    # Execute with progress bar
    for idx, row, task in tqdm(tasks, desc="Labeling examples"):
        label, confidence, reasoning = await task
        
        labeled_data.append({
            'id': row.get('id', f'sample_{idx}'),
            'question': row['question'],
            'answer': row['predicted_answer'],
            'gold_answer': row.get('gold_answer', ''),
            'context': row.get('contexts', []),
            'label': label,
            'confidence': confidence,
            'reasoning': reasoning
        })
    
    return labeled_data

In [ ]:
# Run labeling
print("Starting LLM-as-judge labeling...")
print(f"Processing {len(df_sample)} examples")
print("This may take 30-60 minutes depending on API rate limits")

labeled_data = await label_batch_async(df_sample, max_concurrent=10)

print(f"\n✅ Labeled {len(labeled_data)} examples")

## 6. Quality Filtering and Balancing

In [ ]:
# Convert to DataFrame
df_labeled = pd.DataFrame(labeled_data)

print(f"Total labeled samples: {len(df_labeled)}")
print(f"\nLabel distribution (before filtering):")
print(df_labeled['label'].value_counts())
print(f"\nConfidence statistics:")
print(df_labeled['confidence'].describe())

In [ ]:
# Filter by confidence threshold
confidence_threshold = 0.7
df_filtered = df_labeled[df_labeled['confidence'] >= confidence_threshold].copy()

print(f"After confidence filtering (>= {confidence_threshold}):")
print(f"  Remaining samples: {len(df_filtered)}")
print(f"\nLabel distribution (after filtering):")
print(df_filtered['label'].value_counts())

In [ ]:
# Balance dataset to ~50% correct, ~50% incorrect
correct_samples = df_filtered[df_filtered['label'] == 1]
incorrect_samples = df_filtered[df_filtered['label'] == 0]

print(f"\nCorrect samples: {len(correct_samples)}")
print(f"Incorrect samples: {len(incorrect_samples)}")

# Balance by taking equal numbers
target_per_class = min(len(correct_samples), len(incorrect_samples), 500)

correct_balanced = correct_samples.sample(n=target_per_class, random_state=42)
incorrect_balanced = incorrect_samples.sample(n=target_per_class, random_state=42)

df_balanced = pd.concat([correct_balanced, incorrect_balanced], ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle

print(f"\nBalanced dataset:")
print(f"  Total samples: {len(df_balanced)}")
print(f"  Correct: {(df_balanced['label'] == 1).sum()}")
print(f"  Incorrect: {(df_balanced['label'] == 0).sum()}")

## 7. Manual Verification (Sample)

In [ ]:
# Display a few examples for manual verification
print("Sample labeled examples for verification:\n")

for i in range(min(5, len(df_balanced))):
    row = df_balanced.iloc[i]
    print(f"{'='*80}")
    print(f"Example {i+1}")
    print(f"{'='*80}")
    print(f"Question: {row['question']}")
    print(f"\nGenerated Answer: {row['answer'][:200]}...")
    print(f"\nLabel: {row['label']} ({'CORRECT' if row['label'] == 1 else 'INCORRECT'})")
    print(f"Confidence: {row['confidence']:.2f}")
    print(f"Reasoning: {row['reasoning']}")
    print()

## 8. Save Synthetic Labeled Dataset

In [ ]:
# Save to CSV
output_dir = Path("../data/asqa")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "synthetic_labeled_train.csv"
df_balanced.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ Saved synthetic labeled dataset: {output_path}")
print(f"   Total samples: {len(df_balanced)}")
print(f"   Correct samples: {(df_balanced['label'] == 1).sum()}")
print(f"   Incorrect samples: {(df_balanced['label'] == 0).sum()}")

## 9. Dataset Statistics

In [ ]:
# Analyze dataset statistics
print("Dataset Statistics:\n")

print(f"Total samples: {len(df_balanced)}")
print(f"\nLabel distribution:")
print(df_balanced['label'].value_counts())
print(f"\nConfidence distribution:")
print(df_balanced['confidence'].describe())

# Answer length statistics
df_balanced['answer_length'] = df_balanced['answer'].str.split().str.len()
print(f"\nAnswer length (words):")
print(df_balanced['answer_length'].describe())

# Question length statistics
df_balanced['question_length'] = df_balanced['question'].str.split().str.len()
print(f"\nQuestion length (words):")
print(df_balanced['question_length'].describe())

## 10. Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Label distribution
df_balanced['label'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['red', 'green'])
axes[0, 0].set_title('Label Distribution')
axes[0, 0].set_xlabel('Label (0=Incorrect, 1=Correct)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_xticklabels(['Incorrect', 'Correct'], rotation=0)

# Confidence distribution
df_balanced['confidence'].hist(bins=20, ax=axes[0, 1], edgecolor='black')
axes[0, 1].set_title('Confidence Distribution')
axes[0, 1].set_xlabel('Confidence')
axes[0, 1].set_ylabel('Count')

# Answer length by label
df_balanced.boxplot(column='answer_length', by='label', ax=axes[1, 0])
axes[1, 0].set_title('Answer Length by Label')
axes[1, 0].set_xlabel('Label (0=Incorrect, 1=Correct)')
axes[1, 0].set_ylabel('Answer Length (words)')

# Confidence by label
df_balanced.boxplot(column='confidence', by='label', ax=axes[1, 1])
axes[1, 1].set_title('Confidence by Label')
axes[1, 1].set_xlabel('Label (0=Incorrect, 1=Correct)')
axes[1, 1].set_ylabel('Confidence')

plt.tight_layout()
plt.savefig('../results/synthetic_data_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved visualization to results/synthetic_data_statistics.png")

## Summary

Successfully generated synthetic labeled dataset:
- **Total samples**: ~1000 (balanced)
- **Labeling method**: LLM-as-judge (Gemini 2.5 Flash)
- **Quality control**: Confidence threshold filtering
- **Balance**: ~50% correct, ~50% incorrect

This dataset can be used for:
1. Evaluating filter effectiveness
2. Training evaluation models
3. Ground truth for comprehensive evaluation